##BronzeWork Increamntal
Increamntal Brinze ingestion with re-run safe with watermark logic.

#####Step -1 imports and setup 
import pyspark and delta helpers used in the notebook, switches to the correct caatlog and makes sure Bronze sceham exists before we start loading data

In [0]:
from pyspark.sql import *
from pyspark.sql import functions as f
from delta.tables import DeltaTable 
from pyspark.sql.types import *
import uuid


In [0]:
spark.sql("USE CATALOG `novacart_ecomdatabricks`")

In [0]:
## we cannot create the schema in a forign catolog , like `novacart_ecomdb-catalog` as it came from sql server .
## we need to create our own unity catalog and create a schema there.

spark.sql("CREATE SCHEMA IF NOT EXISTS Bronze_Schema")

#### Step:2 - BRONZE Control Table
the table stores watermark fro each source table .
it helps pipleine to rember :
- the latest timestamp already processed
- the latest primary key processed at that timestamp
- how amny rows were written in the latest run

this is what makes the Bronze load increamntal and rerun safe 


In [0]:
spark.sql("""
          Create Table if not exists novacart_ecomdatabricks.bronze_schema.ingestion_control(
              layer string,
              table_name string,
              ts_col string ,
              pk_col string,
              last_sucessful_ts timestamp,
              last_sucessful_pk bigint,
              last_run_id string,
              rows_written bigint,
              run_status string ,
              updated_at timestamp
              ) Using delta 
          
          """)

##Step3: Source Table configurations

source table will be loaded into bronze and which column should be used as 
- primary key 
- timestamp/ watermark column

it also creates a unique bronze_run_id for current pipeline run


In [0]:
# we have 3 table (orders, products and payments )


tables_cofig = {

    "orders" : {"pk_col": "order_id", "ts_col" : "updated_at"} ,
    "products" : {"pk_col": "product_id", "ts_col" : "updated_at"} ,
    "payments" : {"pk_col": "payment_id", "ts_col" : "processed_at"} 

}

bronze_run_id =str(uuid.uuid4())
print("current_time",bronze_run_id)

##Step4: Helper functions

it contains reusable functions:
 - get_last_sucessful_watermark() reads the altest processed watermark from the control table 
 - upsert_bronze_control updates the control table after a suessful bronze load

 these functions keep the main logic cleaner and easier to understand 

In [0]:
## we will filter the recordss and also ordeby updated date to get the latest track of insertion date.
def get_last_sucessful_watermark(table_name:str):
     cntrl= (
         spark.table("novacart_ecomdatabricks.bronze_schema.ingestion_control")
         .filter(
             (f.col("layer")=="bronze")&
             (f.col("table_name")==table_name)&
             (f.col("run_status")=="success")
             )
             .orderBy(f.col("updated_at").desc())
             .limit(1)
     )
     rows= cntrl.collect()  ## we need to colect in python memory , it will conatin 0 row , taht menas no data is updated or inserted, if 1 that means inserted or updated 
     if not rows:
         return None,None
     return rows[0]["last_sucessful_ts"], rows[0]["last_sucessful_pk"]





In [0]:
from datetime import datetime

def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):

    control_df= spark.createDataFrame(
        [(
            "bronze", 
             table_name,
             ts_col,
             pk_col,
             last_ts,
             int(last_pk) if last_pk is not None else None,
             run_id,
             int(rows_written),
             "success",
             datetime.utcnow()
            )],
    
        schema = """
        layer string,
        table_name string,
        ts_col string ,
        pk_col string,
        last_sucessful_ts timestamp,
        last_sucessful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )
    # Load the Delta table for upsert operations
    dt= DeltaTable.forName(spark,"novacart_ecomdatabricks.bronze_schema.ingestion_control")
    (dt.alias("t")
     .merge (control_df.alias("s"),"t.table_name=s.table_name and t.ts_col=s.ts_col")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute()
    )


##Step:5 -Bronze incremantal load loop 

This is a main Bronze Logic.

for each table , the notebbok :
 1. reads the last watermark.
 2. reads the source table Sql table 
 3. filters only new/chnaged rows.
 4. add bronze audit columns
 5. appends the rows into bronze delta table 
 6. updates the control table

this is the core increamntal logic .


In [0]:
for table_name, cfg in tables_cofig.items():
    ts_col = cfg["ts_col"]
    pk_col = cfg["pk_col"]
    source_table = f"`novacart_ecomdb-catalog`.dbo.{table_name}"
    target_table = f"novacart_ecomdatabricks.bronze_schema.{table_name}_raw"
    last_successful_ts,last_successful_pk = get_last_sucessful_watermark(table_name)
    print(f"Processing table {table_name}")
    print(f"Last successful ts: {last_successful_ts}")
    print(f"Last successful pk: {last_successful_pk}")
    print(f"Source table: {source_table}")
    print(f"Target table: {target_table}")
   
    source_df = spark.read.table(source_table).withColumn(ts_col,f.col(ts_col).cast("timestamp"))
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (f.col(ts_col) > f.lit(last_successful_ts)) |
            (
                (f.col(ts_col) == f.lit(last_successful_ts)) &
                (f.col(pk_col).cast("long") > f.lit(last_successful_pk))
            )
        )

    rows_to_load = (rows_to_load
                   .withColumn("bronze_ingested_at",f.current_timestamp())
                   .withColumn("bronze_run_id",f.lit(bronze_run_id))
                   .withColumn("bronze_source_table",f.lit(source_table))
    )

    rows_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {rows_count}")

    if rows_count == 0:
        print(f"No new rows for {table_name}")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_pk,
            last_successful_ts,
            rows_count,
            bronze_run_id
        )
        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    max_ts = rows_to_load.agg(f.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
    max_pk = (
        rows_to_load
        .filter(f.col(ts_col) == f.lit(max_ts))
        .agg(f.max(pk_col).cast("long").alias("max_pk"))
        .collect()[0]["max_pk"]
    )
    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        rows_count,
        bronze_run_id)

    print(f"Wrote {rows_count} rows to {target_table}")

## Step 6 - Quick validation

final cell prints the bronze row counts nad display table control and verify the increamnetal logic .

In [0]:
## display count:

print("Orders Bronze Count", spark.sql("select count(*) from novacart_ecomdatabricks.bronze_schema.orders_raw").collect()[0][0])

print("Products Bronze Count", spark.sql("select count(*) from novacart_ecomdatabricks.bronze_schema.products_raw").collect()[0][0])

print("Payments Bronze Count", spark.sql("select count(*) from novacart_ecomdatabricks.bronze_schema.payments_raw").collect()[0][0])


display(spark.sql("select * from  novacart_ecomdatabricks.bronze_schema.ingestion_control").orderBy("table_name"))

